## Ana Paula

In [2]:
import pandas as pd
df = pd.read_csv('data/fake_job_postings.csv')
print(df.shape)
print(df.head())

(17880, 18)
   job_id                                      title            location  \
0       1                           Marketing Intern    US, NY, New York   
1       2  Customer Service - Cloud Video Production      NZ, , Auckland   
2       3    Commissioning Machinery Assistant (CMA)       US, IA, Wever   
3       4          Account Executive - Washington DC  US, DC, Washington   
4       5                        Bill Review Manager  US, FL, Fort Worth   

  department salary_range                                    company_profile  \
0  Marketing          NaN  We're Food52, and we've created a groundbreaki...   
1    Success          NaN  90 Seconds, the worlds Cloud Video Production ...   
2        NaN          NaN  Valor Services provides Workforce Solutions th...   
3      Sales          NaN  Our passion for improving quality of life thro...   
4        NaN          NaN  SpotSource Solutions LLC is a Global Human Cap...   

                                         descripti

In [3]:
# Eliminar columnas que no aportan información al modelo
columnas_a_eliminar = ['job_id'] # El ID es solo un contador
df = df.drop(columns=columnas_a_eliminar)

In [4]:
# Rellenar nulos en columnas categóricas y de texto
categoricas_y_texto = ['department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'employment_type', 'required_experience', 'required_education', 'industry', 'function']

for col in categoricas_y_texto:
    df[col] = df[col].fillna('Unspecified')

In [5]:
import re

def limpiar_texto(texto):
    if pd.isna(texto) or texto == 'Unspecified':
        return ''
    texto = str(texto).lower() # Todo a minúsculas
    texto = re.sub(r'[^a-z0-9\s]', '', texto) # Quitar caracteres especiales (deja solo letras, números y espacios)
    texto = re.sub(r'\s+', ' ', texto).strip() # Limpiar espacios dobles o saltos de línea
    return texto

# Ejemplo: Aplicar la limpieza a la columna 'description'
df['description_clean'] = df['description'].apply(limpiar_texto)

In [6]:
# Extraer el código del país (los dos primeros caracteres antes de la primera coma)
df['country'] = df['location'].apply(lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unspecified')

In [7]:
# Ver la distribución de clases
print(df['fraudulent'].value_counts())
print(df['fraudulent'].value_counts(normalize=True) * 100)

fraudulent
0    17014
1      866
Name: count, dtype: int64
fraudulent
0    95.1566
1     4.8434
Name: proportion, dtype: float64


In [8]:
# 1. Comprobar que ya no quedan nulos en las columnas que vas a usar
print(df[['title', 'description_clean', 'country', 'fraudulent']].isnull().sum())

# 2. Ver cómo ha quedado una fila real tras la limpieza
print(df['description_clean'].iloc[0][:300]) # Muestra los primeros 300 caracteres de la primera descripción

title                0
description_clean    0
country              0
fraudulent           0
dtype: int64
food52 a fastgrowing james beard awardwinning online food community and crowdsourced and curated recipe hub is currently interviewing full and parttime unpaid interns to work in a small team of editors executives and developers in its new york city headquartersreproducing andor repackaging existing 


In [9]:
print(df.columns)

Index(['title', 'location', 'department', 'salary_range', 'company_profile',
       'description', 'requirements', 'benefits', 'telecommuting',
       'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent', 'description_clean', 'country'],
      dtype='str')


In [10]:
import pandas as pd
import numpy as np
import re
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


# ==========================================
# 2. INGENIERÍA DE FEATURES PARA EL MODELO
# ==========================================
# Extraemos también el estado y la ciudad de forma segura si 'location' existe
if 'location' in df.columns:
    df['state'] = df['location'].apply(lambda x: str(x).split(',')[1].strip() if pd.notna(x) and len(str(x).split(',')) > 1 else 'Unspecified')
    df['city'] = df['location'].apply(lambda x: str(x).split(',')[2].strip() if pd.notna(x) and len(str(x).split(',')) > 2 else 'Unspecified')
    df.drop(columns=['location'], inplace=True, errors='ignore')

# Creamos métricas numéricas del texto
df['text_length'] = df['description_clean'].apply(len)
df['has_requirements'] = df['requirements'].apply(lambda x: 0 if x == 'Unspecified' else 1)

# Creamos la súper-descripción uniendo tus textos para el NLP de CatBoost
df['full_text'] = df['title'].astype(str) + " " + df['company_profile'].astype(str) + " " + df['description_clean'].astype(str)

# ==========================================
# 3. SELECCIÓN DE VARIABLES Y SPLIT
# ==========================================
# Definimos las columnas que va a usar el modelo
categorical_features = ['country', 'state', 'city', 'employment_type', 'required_experience', 'required_education', 'industry', 'function']
numerical_features = ['telecommuting', 'has_company_logo', 'has_questions', 'text_length', 'has_requirements']
text_feature = 'full_text'

# Nos aseguramos de que todas las columnas seleccionadas realmente existan en el DataFrame
categorical_features = [col for col in categorical_features if col in df.columns]
columnas_totales = categorical_features + numerical_features + [text_feature]

X = df[columnas_totales]
y = df['fraudulent']

# Split estratificado para mantener la proporción de fraudes
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# ==========================================
# 4. ENTRENAMIENTO DE CATBOOST
# ==========================================
# Calculamos el ratio para combatir el fuerte desbalanceo de clases
ratio_desbalanceo = y_train.value_counts()[0] / y_train.value_counts()[1]

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    scale_pos_weight=ratio_desbalanceo,  # Penaliza los fallos en ofertas falsas
    task_type="CPU",
    cat_features=categorical_features,   # Manejo nativo de categorías
    text_features=[text_feature],        # Manejo nativo de texto (NLP)
    early_stopping_rounds=50,
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_test, y_test))

# ==========================================
# 5. EVALUACIÓN DE RESULTADOS
# ==========================================
preds = model.predict(X_test)

print("\n=== MATRIZ DE CONFUSIÓN ===")
print(confusion_matrix(y_test, preds))

print("\n=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, preds))

0:	learn: 0.6725065	test: 0.6759824	best: 0.6759824 (0)	total: 316ms	remaining: 5m 15s
100:	learn: 0.2129106	test: 0.2298310	best: 0.2298310 (100)	total: 17.6s	remaining: 2m 36s
200:	learn: 0.1393623	test: 0.1705242	best: 0.1705242 (200)	total: 35.4s	remaining: 2m 20s
300:	learn: 0.0866597	test: 0.1434207	best: 0.1434207 (300)	total: 55s	remaining: 2m 7s
400:	learn: 0.0587675	test: 0.1354076	best: 0.1344393 (386)	total: 1m 29s	remaining: 2m 14s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.1344392589
bestIteration = 386

Shrink model to first 387 iterations.

=== MATRIZ DE CONFUSIÓN ===
[[3324   79]
 [  11  162]]

=== REPORTE DE CLASIFICACIÓN ===
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      3403
           1       0.67      0.94      0.78       173

    accuracy                           0.97      3576
   macro avg       0.83      0.96      0.88      3576
weighted avg       0.98      0.97      0.98      35